# 🏀 March Madness Fantasy Basketball Projections
**Pipeline:** RealGM stats CSV → Fantasy PPG → Monte Carlo expected games → Ranked draft board

---

## 📋 Before You Start — Build Your RealGM CSV


**Steps:**
1. Go to the RealGM NCAA stats page:  
   👉 https://basketball.realgm.com/ncaa/stats/2026/Averages/Qualified/All/Conference_Regular_Season/All/points/desc/1/
2. Select and copy all **data rows** from the table (skip the header row)
3. Paste into Excel or Google Sheets
4. Navigate to page 2, copy rows, paste below — repeat through all 26 pages
5. Add this as the **very first row** of your spreadsheet:
   ```
   Rank,Player,Team,GP,MPG,PPG,FGM,FGA,FG%,3PM,3PA,3P%,FTM,FTA,FT%,ORB,DRB,RPG,APG,SPG,BPG,TOV,PF
   ```
6. Export/save as **`realgm_stats.csv`** in the same folder as this notebook

> ⚠️ RealGM uses short team **abbreviations** (e.g. `AUB`, `DUKE`, `BYU`) instead of full names.  
> The notebook handles this automatically using a lookup table.

In [132]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
import matplotlib.pylab as plt
import pandas as pd
import numpy as np
import re
from difflib import SequenceMatcher
import warnings
warnings.filterwarnings('ignore')

In [133]:
# Fantasy scoring weights
scoring = {
    'PPG': 1.0,
    'RPG': 1.0,
    'APG': 1.0,
    # 'SPG': 2.0,
    # 'BPG': 2.0,
    # '3PM': 0.5,
}

# Player stats filename
stats_file = 'realgm_stats.csv'

# Path to your Monte Carlo CSV/Excel file
# Expected format: two columns — 'Team' and 'ExpectedGames'
monte_carlo_file = 'expected_games.csv'

# Minimum games and minutes per game to include a player
min_games = 1
min_mpg = 10

# How many players to show in the final draft board
draft_board_size = 350

In [134]:
# real GM team abbreviations mapped to team name
# add new entries as needed
realgm_abbrev = {
    'AIRF':   'Air Force',
    'ALA':    'Alabama',
    'ALCN':   'Alcorn State',
    'ALST':   'Alabama State',
    'AMER':   'American',
    'ARIZ':   'Arizona',
    'ARZ':   'Arizona',
    'ARK':    'Arkansas',
    'ARST':   'Arkansas-Pine Bluff',
    'ASU':    'Arizona State',
    'AUB':    'Auburn',
    'BAY':    'Baylor',
    'BC':     'Boston College',
    'BCST':   'Bethune-Cookman',
    'BOIS':   'Boise State',
    'BRY':    'Bryant',
    'BUF':    'Buffalo',
    'BYU':    'BYU',
    'BYU': 'Brigham Young',
    'CALBU': 'California Baptist',
    'CHAT':   'Chattanooga',
    'CIN':    'Cincinnati',
    'CLEM':   'Clemson',
    'COLG':   'Colgate',
    'COLO':   'Colorado',
    'COPN':   'Coppin State',
    'COST':   'Colorado State',
    'CREI':   'Creighton',
    'CSU':    'Colorado State',
    'DAVI':   'Davidson',
    'DAY':    'Dayton',
    'DELT':   'Delaware State',
    'DRAKE':  'Drake',
    'DUKE':   'Duke',
    'ETSU':   'East Tennessee State',
    'FLA':    'Florida',
    'FLAM':   'Florida A&M',
    'FRES':   'Fresno State',
    'FURD':   'Fordham',
    'FURM':   'Furman',
    'GONZ':   'Gonzaga',
    'GRAM':   'Grambling',
    'GW':     'George Washington',
    'HAW':    'Hawaii',
    'HOF':   'Hofstra',
    'HOUS':   'Houston',
    'HOW':    'Howard',
    'HP':     'High Point',
    'HPT':    'High Point',
    'IAST':   'Iowa State',
    'ILL':    'Illinois',
    'IND':    'Indiana',
    'IOWA':   'Iowa',
    'IWAST':  'Iowa State',
    'JKSN':   'Jackson State',
    'JMU':    'James Madison',
    'KAN':    'Kansas',
    'KANS':   'Kansas',
    'KAST':   'Kansas State',
    'KEN':    'Kentucky',
    'KU':     'Kansas',
    'LIB':    'Liberty',
    'LIP':    'Lipscomb',
    'LMU':    'Loyola Marymount',
    'LONG':   'Longwood',
    'LOU':    'Louisville',
    'LSU':    'LSU',
    'LVILL':  'Louisville',
    'MARQ':   'Marquette',
    'MCN':    'McNeese',
    'MD':     'Maryland',
    'MEM':    'Memphis',
    'MERR':   'Merrimack',
    'MICH':   'Michigan',
    'MINN':   'Minnesota',
    'MISS':   'Ole Miss',
    'MIZ':    'Missouri',
    'MONT':   'Montana',
    'MORE':   'Morehead State',
    'MORG':   'Morgan State',
    'MSM':    "Mount St. Mary's",
    'MSST':   'Mississippi State',
    'MCHST':    'Michigan State',
    'MVST':   'Mississippi Valley State',
    'MZZOU':  'Missouri',
    'NCAT':   'North Carolina A&T',
    'NCST':   'NC State',
    'ND':     'Notre Dame',
    'NDSU': 'North Dakota State',
    'NEB':    'Nebraska',
    'NFST':   'Norfolk State',
    'NMST':   'New Mexico State',
    'NOVA':   'Villanova',
    'NW':     'Northwestern',
    'NWST':   'Northwestern',
    'OKLA':   'Oklahoma',
    'OKST':   'Oklahoma State',
    'ORAL':   'Oral Roberts',
    'ORE':    'Oregon',
    'OSU':    'Ohio State',
    'PACF':   'Pacific',
    'PENNST':   'Penn State',
    'PENN': 'Pennsylvania',
    'PEPP':   'Pepperdine',
    'PITT':   'Pittsburgh',
    'PORT':   'Portland',
    'PRST':   'Prairie View A&M',
    'PUR':    'Purdue',
    'PURD':   'Purdue',
    'RICH':   'Richmond',
    'RMU':    'Robert Morris',
    'RUTG':   'Rutgers',
    'SAMF':   'Samford',
    'SCST':   'South Carolina State',
    'SCU':    'Santa Clara',
    'SDST':   'San Diego State',
    'SDSU':   'San Diego State',
    'SEAT':   'Seattle',
    'SFU':    'Saint Francis',
    'SIUE':   'SIU Edwardsville',
    'SJST':   'San Jose State',
    'SMCA':   "Saint Mary's",
    'STMRY':  "Saint Mary's",
    'STJ': "St. John's (NY)",
    'SUST':   'Southern',
    'SYRA':   'Syracuse',
    'TCU':    'TCU',
    'TENN':   'Tennessee',
    'TEX':    'Texas',
    'TROY':   'Troy',
    'TTU':    'Texas Tech',
    'TXAM':   'Texas A&M',
    'TXSO':   'Texas Southern',
    'TXST':   'Texas State',
    'TXTCH':  'Texas Tech',
    'UA':     'Alabama',
    'UCF':    'UCF',
    'UCI':    'UC Irvine',
    'UCLA':   'UCLA',
    'UCON':   'UConn',
    'UConn':   'Connecticut',
    'UCSB':   'UC Santa Barbara',
    'UCSD':   'UC San Diego',
    'UGA':    'Georgia',
    'UK':     'Kentucky',
    'UNC':    'North Carolina',
    'UNI': 'Northern Iowa',
    'UNL': 'Nebraska',
    'UNLV':   'UNLV',
    'UNM':    'New Mexico',
    'UNR':    'Nevada',
    'USC':    'USC',
    'USD':    'San Diego',
    'USF':    'South Florida',
    'USOFL':    'South Florida',
    'USU':    'Utah State',
    'UTAH':   'Utah',
    'UVA':   'Virginia',
    'U OF H': 'Houston',
    'U of H': 'Houston',
    'U OF T': 'Toledo',
    'U of T': 'Toledo',
    'UWISC':  'Wisconsin',
    'VAN':    'Vanderbilt',
    'VANDY':  'Vanderbilt',
    'VCU':    'VCU',
    'VCU':    'Virginia Commonwealth',
    'VILL':   'Villanova',
    'WASH':   'Washington',
    'WEST':   'Western Carolina',
    'WF':     'Wake Forest',
    'WIS':    'Wisconsin',
    'WOF':    'Wofford',
    'WSU':    'Washington State',
    'WVU':    'West Virginia',
    'WYO':    'Wyoming',
    'XAV':    'Xavier',
    'YALE':   'Yale',
    'ZAGS':   'Gonzaga',
    'FRMN': 'Furman',
    'LEHI': 'Lehigh',
    'MCNST': 'McNeese State',
    'STMRY': "Saint Mary's (CA)",
    'TAMU': 'Texas A&M',
    'LIU': 'Long Island University',
    'KENN': 'Kennesaw State',
    'MIAMI': 'Miami (FL)',
    'QUC': 'Queens (NC)',
    'UMBC': 'Maryland-Baltimore County',
    'STLOU': 'Saint Louis',
    'AKR': 'Akron',
    'WRST': 'Wright State',
    'SMU': 'Southern Methodist',
    'TNST': 'Tennessee State'
}

In [135]:
with open(stats_file, 'r', encoding='utf-8-sig') as f:
    first_line = f.readline()
delimiter = '\t' if first_line.count('\t') > first_line.count(',') else ','

raw = pd.read_csv(stats_file, sep=delimiter, dtype=str)
raw.columns = raw.columns.str.strip()

raw = raw.loc[:, ~raw.columns.str.match(r'^(Unnamed|#|Rank|Rk)$', case=False)]

col_map = {
    'G': 'GP', 'MIN': 'MPG', 'PTS': 'PPG', 'REB': 'RPG', 'TRB': 'RPG',
    'AST': 'APG', 'STL': 'SPG', 'BLK': 'BPG', '3P': '3PM',
}
raw = raw.rename(columns={c: col_map[c] for c in raw.columns if c in col_map})

if 'Player' in raw.columns:
    raw = raw[~raw['Player'].astype(str).str.strip().isin(['Player', '#', 'Rank', ''])]

for col in [c for c in raw.columns if c not in ('Player', 'Team')]:
    raw[col] = pd.to_numeric(raw[col], errors='coerce')

raw['Player'] = raw['Player'].astype(str).str.strip()
raw['Team'] = raw['Team'].astype(str).str.strip()
raw.dropna(subset=['Player', 'Team'], inplace=True)
raw.drop_duplicates(subset=['Player', 'Team'], inplace=True)
raw.reset_index(drop=True, inplace=True)

print(f'{len(raw):,} players loaded')

2,583 players loaded


In [136]:
abbrev_ci = {k.upper(): v for k, v in realgm_abbrev.items()}
raw['team_full'] = raw['Team'].str.upper().map(abbrev_ci)

unknown = raw[raw['team_full'].isna()]['Team'].unique()
if len(unknown) > 0:
    print(f'{len(unknown)} unknown abbreviation(s) — add to realgm_abbrev:')
    for abbr in sorted(unknown):
        count = (raw['Team'] == abbr).sum()
        
else:
    print('all abbreviations resolved')

raw = raw.rename(columns={'Team': 'team_abbrev'})
raw['Team'] = raw['team_full'].fillna(raw['team_abbrev'])
raw.drop(columns='team_full', inplace=True)

raw[['Player', 'team_abbrev', 'Team']].head()


260 unknown abbreviation(s) — add to realgm_abbrev:


,Player,team_abbrev,Team
0,Daeshun Ruffin,JKSN,Jackson State
1,A.J. Dybantsa,BYU,Brigham Young
2,Jordan Riley,ECAR,ECAR
3,"Darius Acuff, Jr.",ARK,Arkansas
4,Dontae Horne,PVAMU,PVAMU


In [137]:
def load_mc(path):
    mc = pd.read_excel(path) if path.lower().endswith(('xlsx', 'xls')) else pd.read_csv(path)
    mc.columns = mc.columns.str.strip()
    remap = {}
    for c in mc.columns:
        cl = c.lower().replace(' ', '_')
        if cl in ('team', 'school', 'program'):
            remap[c] = 'Team'
        elif cl in ('expected_games', 'exp_games', 'expectedgames', 'games', 'avg_games', 'projected_games'):
            remap[c] = 'ExpectedGames'
        elif cl in ('seed', 'seeding'):
            remap[c] = 'Seed'
    mc = mc.rename(columns=remap)
    for req in ('Team', 'ExpectedGames'):
        if req not in mc.columns:
            raise ValueError(f"missing column '{req}', found: {mc.columns.tolist()}")
    mc['ExpectedGames'] = pd.to_numeric(mc['ExpectedGames'], errors='coerce')
    mc['Team'] = mc['Team'].astype(str).str.strip()
    return mc

def normalize(name):
    name = re.sub(r'[^a-z0-9 ]', ' ', name.lower())
    return re.sub(r'\s+', ' ', name).strip()

mc_df = load_mc(monte_carlo_file)
print(f'{len(mc_df)} teams  |  xGames range: {mc_df.ExpectedGames.min():.2f} – {mc_df.ExpectedGames.max():.2f}')

expanded_teams = list(raw['Team'].unique())
expanded_lower = {t.lower(): t for t in expanded_teams}
expanded_norm  = {normalize(t): t for t in expanded_teams}

auto_fixed = {}
still_missing = []

for team in mc_df['Team']:
    if team.lower() in expanded_lower:
        if expanded_lower[team.lower()] != team:
            auto_fixed[team] = expanded_lower[team.lower()]
    elif normalize(team) in expanded_norm:
        auto_fixed[team] = expanded_norm[normalize(team)]
    else:
        still_missing.append(team)

if auto_fixed:
    mc_df['Team'] = mc_df['Team'].replace(auto_fixed)
    print(f'\nauto-resolved {len(auto_fixed)} name difference(s):')
    for old, new in auto_fixed.items():
        print(f'  "{old}" → "{new}"')

if still_missing:
    print(f'\ncould not match {len(still_missing)} team(s):')
    for t in still_missing:
        scored = sorted([(SequenceMatcher(None, normalize(t), normalize(e)).ratio(), e) for e in expanded_teams], reverse=True)[:3]
        sugg = [f'"{e}" ({r:.2f})' for r, e in scored if r > 0.5]
        print(f'  "{t}" → suggestions: {sugg or "none"}')
    print('\nadd the abbrev to realgm_abbrev and re-run')
else:
    print('\nall MC teams matched')

64 teams  |  xGames range: 1.00 – 4.70

auto-resolved 2 name difference(s):
  "Siena" → "SIENA"
  "Idaho" → "IDAHO"

all MC teams matched


In [138]:
df = raw.copy()

if 'MPG' in df.columns:
    df = df[df['MPG'] >= min_mpg]
if 'GP' in df.columns:
    df = df[df['GP'] >= min_games]

tourney_teams = set(mc_df['Team'].str.lower())
df['_key'] = df['Team'].str.lower().str.strip()
tourney = df[df['_key'].isin(tourney_teams)].drop(columns='_key').copy()

print(f'{len(raw):,} → {len(df):,} after stat filters → {len(tourney):,} tournament players across {tourney["Team"].nunique()} teams')

empty = [t for t in mc_df['Team'] if t.lower() not in set(tourney['Team'].str.lower())]
if empty:
    print(f'\nno players matched for: {empty}')
    print('probably still a name mismatch — check cell 6')

tourney.head(5)

2,583 → 2,576 after stat filters → 461 tournament players across 64 teams


,Player,team_abbrev,GP,MPG,PPG,FGM,FGA,FG%,3:00 PM,3PA,...,FT%,ORB,DRB,RPG,APG,SPG,BPG,TOV,PF,Team
1,A.J. Dybantsa,BYU,18,36.3,25.9,8.9,18.9,0.471,1.8,5.2,...,0.740,1.3,5.1,6.4,3.8,0.8,0.3,3.3,1.5,Brigham Young
3,"Darius Acuff, Jr.",ARK,17,36.5,24.8,8.5,17.4,0.492,2.5,5.7,...,0.826,0.4,2.6,3.0,6.6,0.7,0.4,2.1,1.5,Arkansas
5,"Dominique Daniels, Jr.",CALBU,18,36.0,23.8,8.0,18.3,0.436,1.8,5.5,...,0.775,0.6,2.8,3.4,2.9,0.8,0.1,2.3,1.8,California Baptist
15,Cameron Boozer,DUKE,18,33.9,22.7,7.9,13.4,0.593,1.6,3.5,...,0.795,3.4,6.9,10.3,3.9,1.6,0.2,2.3,1.3,Duke
17,J.T. Toppin,TXTCH,13,35.7,22.6,9.6,17.8,0.541,0.9,2.6,...,0.780,3.8,7.3,11.1,2.4,1.5,1.8,2.3,2.5,Texas Tech


In [139]:
missing = [s for s in scoring if s not in tourney.columns]
if missing:
    raise ValueError(f'scoring columns not in data: {missing}')

tourney['FPPG'] = tourney.apply(
    lambda r: sum(float(r.get(s, 0) or 0) * w for s, w in scoring.items()), axis=1
).round(2)

print(f'formula: {" + ".join(f"{w}×{s}" for s, w in scoring.items())}')
print()
cols = ['Player', 'Team', 'GP', 'MPG'] + list(scoring.keys()) + ['FPPG']
tourney.nlargest(10, 'FPPG')[[c for c in cols if c in tourney.columns]]


formula: 1.0×PPG + 1.0×RPG + 1.0×APG



,Player,Team,GP,MPG,PPG,RPG,APG,FPPG
15,Cameron Boozer,Duke,18,33.9,22.7,10.3,3.9,36.9
1,A.J. Dybantsa,Brigham Young,18,36.3,25.9,6.4,3.8,36.1
17,J.T. Toppin,Texas Tech,13,35.7,22.6,11.1,2.4,36.1
3,"Darius Acuff, Jr.",Arkansas,17,36.5,24.8,3.0,6.6,34.4
25,Graham Ike,Gonzaga,15,32.3,21.9,8.2,2.1,32.2
53,Caleb Wilson,North Carolina,11,33.5,20.1,7.7,3.1,30.9
28,Tyler Tanner,Vanderbilt,18,36.2,21.5,3.4,5.5,30.4
132,"Christian Anderson, Jr.",Texas Tech,17,39.2,18.1,3.8,8.3,30.2
5,"Dominique Daniels, Jr.",California Baptist,18,36.0,23.8,3.4,2.9,30.1
43,Nick Boyd,Wisconsin,20,32.9,20.8,3.9,4.5,29.2


In [140]:
mc_m = mc_df.copy()
mc_m['_key'] = mc_m['Team'].str.lower().str.strip()
tourney['_key'] = tourney['Team'].str.lower().str.strip()
seed_cols = ['Seed'] if 'Seed' in mc_m.columns else []

proj = tourney.merge(
    mc_m[['_key', 'ExpectedGames'] + seed_cols], on='_key', how='left'
).drop(columns='_key')

proj['ProjectedFP'] = (proj['FPPG'] * proj['ExpectedGames']).round(2)
print(f'{len(proj)} players with projections')
proj[['Player', 'Team', 'FPPG', 'ExpectedGames', 'ProjectedFP']].head(5)

461 players with projections


,Player,Team,FPPG,ExpectedGames,ProjectedFP
0,A.J. Dybantsa,Brigham Young,36.1,1.9342,69.82
1,"Darius Acuff, Jr.",Arkansas,34.4,2.8161,96.87
2,"Dominique Daniels, Jr.",California Baptist,30.1,1.0361,31.19
3,Cameron Boozer,Duke,36.9,4.6803,172.70
4,J.T. Toppin,Texas Tech,36.1,2.4408,88.11


In [141]:
cols = (
    (['Seed'] if 'Seed' in proj.columns else []) +
    ['Player', 'Team', 'team_abbrev', 'GP', 'MPG'] +
    list(scoring.keys()) +
    ['FPPG', 'ExpectedGames', 'ProjectedFP']
)
cols = [c for c in cols if c in proj.columns]

draft_board = (
    proj[cols]
    .sort_values('ProjectedFP', ascending=False)
    .head(draft_board_size)
    .reset_index(drop=True)
)
draft_board.index += 1
draft_board.index.name = 'Rank'

fmt = {k: '{:.2f}' for k in ['FPPG', 'ExpectedGames', 'ProjectedFP'] if k in draft_board.columns}
styled = draft_board.style.set_caption('🏀 March Madness Fantasy Draft Board')
for col, cmap in [('ProjectedFP', 'YlGn'), ('FPPG', 'Blues'), ('ExpectedGames', 'Oranges')]:
    if col in draft_board.columns:
        styled = styled.background_gradient(subset=[col], cmap=cmap)
display(styled.format(fmt))

,Seed,Player,Team,team_abbrev,GP,MPG,PPG,RPG,APG,FPPG,ExpectedGames,ProjectedFP
Rank,,,,,,,,,,,,
1,1,Cameron Boozer,Duke,DUKE,18,33.900000,22.700000,10.300000,3.900000,36.90,4.68,172.70
2,1,Yaxel Lendeborg,Michigan,MICH,20,31.500000,14.600000,7.300000,3.300000,25.20,4.70,118.51
3,1,Brayden Burries,Arizona,ARZ,18,31.700000,17.500000,6.200000,2.900000,26.60,4.37,116.37
4,1,"Morez Johnson, Jr.",Michigan,MICH,20,25.700000,13.900000,7.600000,1.100000,22.60,4.70,106.28
5,3,Graham Ike,Gonzaga,ZAGS,15,32.300000,21.900000,8.200000,2.100000,32.20,3.28,105.76
6,3,Keaton Wagler,Illinois,ILL,20,35.500000,19.900000,4.400000,4.900000,29.20,3.59,104.74
7,2,Joshua Jefferson,Iowa State,IWAST,18,33.000000,15.500000,7.700000,5.000000,28.20,3.68,103.78
8,1,Alex Condon,Florida,FLA,18,30.500000,15.100000,6.800000,3.500000,25.40,3.91,99.40
9,2,Braden Smith,Purdue,PURD,20,35.700000,16.500000,3.500000,8.400000,28.40,3.48,98.77


In [142]:
if 'Seed' not in proj.columns:
    print('no Seed column in MC file, skipping sleeper board')
else:
    sleeper_pool = proj[proj['Seed'].notna() & (proj['Seed'].astype(float) >= 8)]

    cols = (
        ['Seed', 'Player', 'Team', 'team_abbrev', 'GP', 'MPG'] +
        list(scoring.keys()) +
        ['FPPG', 'ExpectedGames', 'ProjectedFP']
    )
    cols = [c for c in cols if c in sleeper_pool.columns]

    sleeper_board = (
        sleeper_pool[cols]
        .sort_values('ProjectedFP', ascending=False)
        .head(draft_board_size)
        .reset_index(drop=True)
    )
    sleeper_board.index += 1
    sleeper_board.index.name = 'Rank'

    fmt = {k: '{:.2f}' for k in ['FPPG', 'ExpectedGames', 'ProjectedFP'] if k in sleeper_board.columns}
    styled = sleeper_board.style.set_caption('🎯 Sleeper Board — Seeds 8+')
    for col, cmap in [('ProjectedFP', 'YlGn'), ('FPPG', 'Blues'), ('ExpectedGames', 'Oranges')]:
        if col in sleeper_board.columns:
            styled = styled.background_gradient(subset=[col], cmap=cmap)
    display(styled.format(fmt))


,Seed,Player,Team,team_abbrev,GP,MPG,PPG,RPG,APG,FPPG,ExpectedGames,ProjectedFP
Rank,,,,,,,,,,,,
1,8,Bruce Thornton,Ohio State,OSU,20,36.700000,19.700000,5.400000,4.100000,29.20,1.74,50.71
2,9,Bennett Stirtz,Iowa,IOWA,20,38.400000,22.200000,2.400000,3.800000,28.40,1.75,49.60
3,10,Rashaun Agee,Texas A&M,TAMU,18,30.400000,16.100000,9.300000,2.800000,28.20,1.57,44.22
4,11,Boopie Miller,Southern Methodist,SMU,17,35.500000,18.900000,3.700000,6.200000,28.80,1.47,42.47
5,9,Mason Falslev,Utah State,USU,20,32.600000,16.100000,5.800000,3.200000,25.10,1.62,40.59
6,8,"Juni Mobley, Jr.",Ohio State,OSU,18,32.900000,16.300000,2.600000,2.600000,21.50,1.74,37.33
7,8,Duke Brennan,Villanova,VILL,20,32.400000,12.400000,9.900000,2.000000,24.30,1.53,37.20
8,10,"Mark Mitchell, Jr.",Missouri,MZZOU,18,35.600000,18.400000,4.800000,4.200000,27.40,1.36,37.20
9,8,Devin Royal,Ohio State,OSU,19,33.600000,14.300000,5.400000,1.700000,21.40,1.74,37.16


In [143]:
draft_board.to_csv('fantasy_draft_board.csv')